download from companies marketcap

In [10]:
%load_ext autoreload
%autoreload 2

In [11]:
import requests
import os
from datetime import date

# --- Configuração ---
URL = "https://companiesmarketcap.com/largest-companies-by-revenue/?download=csv"
OUTPUT_DIR = "../data/raw/"
FILE_PREFIX = "top_companies_revenue"
# --------------------

today = date.today().strftime("%Y%m%d")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_{today}.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

with open(OUTPUT_PATH, "wb") as f:
    f.write(response.content)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Tamanho: {len(response.content):,} bytes")


Arquivo salvo em: ../data/raw/top_companies_revenue_20260818.csv
Tamanho: 767,349 bytes


create interim dataset with standarlized columns

In [12]:
import glob
files = sorted(glob.glob("../data/raw/top_companies_revenue_*.csv"))
raw_data_file = files[-1]

print(f"Arquivo mais recente encontrado: {raw_data_file}")

import pandas as pd
df = pd.read_csv(raw_data_file)

std_columns = {
    "Rank": "rank",
    "Name": "company_name",
    "Symbol": "stock_symbol",
    "revenue_ttm": "revenue_ttm",
    "price (USD)": "price_usd",
    "country": "company_country",
}
df = df.rename(columns=std_columns)

replacement_char = {
    ",": ". ",
    ";": ". ",
    ":": ". ",
    "_": "-",
    "[": "(",
    "]": ")",
    "{": "(",
    "}": ")",
    "\"": "",
    "%": " percent ",
    "$": " dollar ",
    "#": " number ",
    "@": " at ",
    "!": "",
    "*": "",
    "+": " plus ",
    "=": " equals ",
    "<": " less than ",
    ">": " greater than ",
    "/": " slash ",
    "\\": " backslash ",
    "|": " pipe ",
    "~": " tilde "
}

for char in replacement_char.keys():
    df["company_name"] = df["company_name"].str.replace(char, replacement_char[char], regex=False)
    df["stock_symbol"] = df["stock_symbol"].str.replace(char, replacement_char[char], regex=False)

df["company_name"] = df["company_name"].str.lower()
display(df)


Arquivo mais recente encontrado: ../data/raw\top_companies_revenue_20260818.csv


,rank,company_name,stock_symbol,revenue_ttm,price_usd,company_country
0,1,amazon,AMZN,742776000000,259.45000,United States
1,2,walmart,WMT,725305000000,115.20000,United States
2,3,saudi aramco,2222.SR,459047318045,7.07969,Saudi Arabia
3,4,apple,AAPL,451442000000,310.03000,United States
4,5,unitedhealth,UNH,449713000000,393.93000,United States
...,...,...,...,...,...,...
11256,11257,kinnevik,KINV-A.ST,-879716521,6.59039,Sweden
11257,11258,burford capital,BUR,-1543511000,4.14000,Guernsey
11258,11259,banque nationale de belgique,BNB.BR,-1768524292,449.23000,Belgium
11259,11260,secure energy services,SES.TO,-2151095178,17.11060,Canada


Get GICS-like classification from FinanceDatabase

In [13]:
import pandas as pd
import financedatabase as fd

equities = fd.Equities()
p = equities.select(
    #only_primary_listing=True
)
fd_dataset = p.reset_index()

In [14]:
# merge the two datasets on stock_symbol, input the following columns from fd_dataset: sector, industry, sub_industry, website, zipcode

companies = pd.merge(
    df,
    fd_dataset[["symbol", "sector", "industry_group", "industry", "website", "zipcode"]],
    how="left",
    left_on="stock_symbol",
    right_on="symbol",
)

companies = pd.merge(df, fd_dataset[["symbol", "sector", "industry_group", "industry", "website", "zipcode"]], how="left", left_on="stock_symbol", right_on="symbol")

# delete the symbol column from companies
companies = companies.drop(columns=["symbol"])


# Create interim directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

today = date.today().strftime("%Y%m%d")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"01_data_after_first_financialdata_{today}.csv")

companies.to_csv(OUTPUT_PATH, index=False)
print(f"File saved to: {OUTPUT_PATH}")

File saved to: ../data/raw/01_data_after_first_financialdata_20260818.csv


In [ ]:
# pegar a primeira ocorrência de uma empresa que não tem website

company_without_website = companies[companies["website"].isnull()].iloc[0]

print(f"Primeira empresa sem website: {company_without_website['company_name']} ({company_without_website['stock_symbol']})")
print(f"Linha completa: {company_without_website.to_dict()}")

Primeira empresa sem website: sinopec (600028.SS)
Linha completa: {'rank': 6, 'company_name': 'sinopec', 'stock_symbol': '600028.SS', 'revenue_ttm': 444749587530, 'price_usd': 0.751388, 'company_country': 'China', 'sector': nan, 'industry_group': nan, 'industry': nan, 'website': nan, 'zipcode': nan}


: 